In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [ ]:
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir = '/content/drive/MyDrive/Pollen Training Images'
csv_file = '/content/image_counts.csv'

df = pd.read_csv(csv_file).dropna()
df['filename'] = df['filename'].apply(lambda x: os.path.join(data_dir, x))

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Create ImageDataGenerator instances
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='count',
    target_size=(150, 150),
    batch_size=10,
    class_mode='raw',
    seed=42,
    subset='training'
)

validation_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filename',
    y_col='count',
    target_size=(150, 150),
    batch_size=10,
    class_mode='raw',
    seed=42
)


print("Train Generator Samples:", train_generator.samples)
print("Validation Generator Samples:", validation_generator.samples)


Found 223 validated image filenames.
Found 56 validated image filenames.
Train Generator Samples: 223
Validation Generator Samples: 56


In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(1)  # No activation function for regression
])

model.compile(
    loss='mse',  # Use mean squared error loss for regression
    optimizer='adam'
)

# Step 5: Training
history = model.fit(
    train_generator,
    steps_per_epoch=5,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=50
)

Epoch 1/10
5/5 [==============================] - ETA: 0s - loss: 127.6003

5/5 [==============================] - 23s 5s/step - loss: 127.6003 - val_loss: 33.4208
Epoch 2/10
5/5 [==============================] - 9s 1s/step - loss: 70.1119
Epoch 3/10
5/5 [==============================] - 9s 2s/step - loss: 60.2462
Epoch 4/10
5/5 [==============================] - 11s 2s/step - loss: 51.1669
Epoch 5/10
5/5 [==============================] - 9s 2s/step - loss: 54.2092
Epoch 6/10
5/5 [==============================] - 9s 2s/step - loss: 46.1426
Epoch 7/10
5/5 [==============================] - 9s 2s/step - loss: 28.0551
Epoch 8/10
5/5 [==============================] - 11s 2s/step - loss: 33.3451
Epoch 9/10
5/5 [==============================] - 11s 2s/step - loss: 33.3690
Epoch 10/10
5/5 [==============================] - 10s 2s/step - loss: 32.8543


In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filename',
    y_col='count',
    target_size=(150, 150),
    batch_size=20,
    class_mode='raw',  # Use 'raw' to return numpy arrays
    seed=42
)

test_loss = model.evaluate(test_generator)
print('Test loss (MSE):', test_loss)

Found 26 validated image filenames.
2/2 [==============================] - 4s 546ms/step - loss: 23.3350
Test loss (MSE): 23.335004806518555


In [ ]:
model.save("/content/trained_model.h5")

In [ ]:
!pip install tensorflowjs

In [ ]:
!tensorflowjs_converter --input_format=keras /content/trained_model.h5 tfjs_model

2024-05-16 02:10:57.515261: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
from google.colab import files
!zip -r tfjs_model.zip tfjs_model
files.download('tfjs_model.zip')

  adding: tfjs_model/ (stored 0%)
  adding: tfjs_model/model.json (deflated 85%)
  adding: tfjs_model/group1-shard3of4.bin (deflated 8%)
  adding: tfjs_model/group1-shard2of4.bin (deflated 8%)
  adding: tfjs_model/group1-shard1of4.bin (deflated 8%)
  adding: tfjs_model/group1-shard4of4.bin (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>